# 🧠 DSA Lab 08: Singly Linked List: Nodes, Pointers and Dynamic Memory
### Student Notebook

**Data Structures & Algorithms Lab Manual | University of Okara | Fall 2026**

Prepared by **Muhammad Ramzan**<br>
MPhil Scholar (Artificial Intelligence), PUCIT, University of the Punjab, Lahore<br>
Lab Engineer, Department of Computer Science, University of Okara

🔗 [GitHub](https://github.com/RamzanPUCIT) · [LinkedIn](https://www.linkedin.com/in/ramzan-ai) · [ResearchGate](https://www.researchgate.net/profile/Muhammad-Ramzan-105)

---
**How to use:** run each code cell with **Shift + Enter**, from top to bottom. Read the matching section in the Lab 08 manual, change values and re-run to see what happens.

> 📝 **Your job:** run the examples, then complete every `# TODO` in the Lab Tasks part. Each task has a **test cell**: when it prints ✅ your solution is correct. Write pseudocode as comments before coding!

## 📖 Part A: Structure of a Linked List

A **singly linked list** is a sequence of **nodes**. Each node has two parts: the **data** and a reference called **next** that points to the following node. The first node is the **head**; the last node’s `next` is **None** (NULL).

**A singly linked list**

```text
  head
   |
   v
+------+---+     +------+---+     +------+---+     +------+------+
|  10  | o-+---->|  20  | o-+---->|  30  | o-+---->|  40  | None |
+------+---+     +------+---+     +------+---+     +------+------+
  data  next                                          ^
                                                     tail
The nodes can be anywhere in memory, e.g. addresses 5000, 1200, 8800, 3400
```

|  | Array | Linked List |
|---|---|---|
| Memory | Contiguous block | Nodes scattered anywhere |
| Size | Fixed or resized by copying | Grows one node at a time |
| Access k-th element | O(1) | O(n), must walk from head |
| Insert / delete at front | O(n), shifting | O(1) |
| Insert / delete at end | O(1) amortised | O(1) with a tail pointer (insert) |
| Insert after a known node | O(n) | O(1) |
| Extra memory | None per element | One reference per node |
| Cache performance | Excellent | Poorer (nodes are far apart) |

## 🧱 Part B: The Node and LinkedList Classes

In [ ]:
class Node:
    def __init__(self, data):
        self.data = data
        self.next = None           # reference to the next node

# Building a list by hand: 10 -> 20 -> 30
a = Node(10); b = Node(20); c = Node(30)
a.next = b
b.next = c
head = a

current = head                     # traversal
while current is not None:
    print(current.data, end=" -> ")
    current = current.next
print("None")

Now we wrap this in a `LinkedList` class that keeps track of `head`, `tail` and `size`, so users of our list never handle nodes directly.

In [ ]:
class LinkedList:
    def __init__(self):
        self.head = None
        self.tail = None
        self.size = 0

    def __len__(self):
        return self.size

    def __str__(self):
        parts, cur = [], self.head
        while cur:
            parts.append(str(cur.data))
            cur = cur.next
        return " -> ".join(parts + ["None"])

    def is_empty(self):
        return self.head is None

## ➕ Part C: Insertion

### Insert at the Head: O(1)

**Steps (order matters!)**

```text
Before:  head -> [20] -> [30] -> None
1. new = Node(10)
2. new.next = head          [10] -> [20] -> [30] -> None
3. head = new               head -> [10] -> [20] -> [30] -> None
```

### Insert at the Tail: O(1) with a tail pointer

Without a tail pointer we would have to walk the whole list to find the last node (O(n)). With `tail`, we simply do `tail.next = node` and then `tail = node`. If the list is empty, the new node is both head and tail.

### Insert at a Position: O(n)

Walk to the node **before** the position (index − 1), then link the new node in between. Always connect the new node to the rest of the list **before** changing the previous node’s `next`, or you will lose the rest of the list.

In [ ]:
def push_front(self, data):
    node = Node(data)
    node.next = self.head
    self.head = node
    if self.tail is None:          # list was empty
        self.tail = node
    self.size += 1

def push_back(self, data):
    node = Node(data)
    if self.head is None:
        self.head = self.tail = node
    else:
        self.tail.next = node
        self.tail = node
    self.size += 1

def insert_at(self, index, data):
    if index < 0 or index > self.size:
        raise IndexError("index out of range")
    if index == 0:
        return self.push_front(data)
    if index == self.size:
        return self.push_back(data)
    prev = self.head
    for _ in range(index - 1):     # walk to the node before index
        prev = prev.next
    node = Node(data)
    node.next = prev.next          # 1. connect new node to the rest
    prev.next = node               # 2. connect previous node to new node
    self.size += 1

LinkedList.push_front = push_front
LinkedList.push_back = push_back
LinkedList.insert_at = insert_at

ll = LinkedList()
for x in [20, 30, 50]:
    ll.push_back(x)
ll.push_front(10)
ll.insert_at(3, 40)
print(ll, "| size =", len(ll))

## 🔎 Part D: Search and Access

In [ ]:
def find(self, target):
    cur, index = self.head, 0
    while cur:
        if cur.data == target:
            return index
        cur = cur.next
        index += 1
    return -1

def get(self, index):
    if not 0 <= index < self.size:
        raise IndexError("index out of range")
    cur = self.head
    for _ in range(index):
        cur = cur.next
    return cur.data

LinkedList.find = find
LinkedList.get = get
print(ll.find(40), ll.find(99), ll.get(2))

## ➖ Part E: Deletion

To delete a node, make the **previous node skip over it**: `prev.next = prev.next.next`. Python’s garbage collector frees the removed node automatically (in C++ you must `delete` it yourself). Special cases: deleting the head, deleting the tail (update `tail`), and an empty list.

**Skipping a node**

```text
Delete 30:   head -> [10] -> [20] -> [30] -> [40] -> None
                          prev    target
prev.next = target.next
             head -> [10] -> [20] ---------> [40] -> None
```

In [ ]:
def pop_front(self):
    if self.head is None:
        raise IndexError("pop from empty list")
    data = self.head.data
    self.head = self.head.next
    if self.head is None:
        self.tail = None
    self.size -= 1
    return data

def remove(self, target):
    """Remove the first node containing target. Return True if removed."""
    if self.head is None:
        return False
    if self.head.data == target:
        self.pop_front()
        return True
    prev = self.head
    while prev.next and prev.next.data != target:
        prev = prev.next
    if prev.next is None:
        return False               # not found
    if prev.next is self.tail:
        self.tail = prev           # deleting the last node
    prev.next = prev.next.next
    self.size -= 1
    return True

LinkedList.pop_front = pop_front
LinkedList.remove = remove
print(ll)
ll.remove(30); ll.remove(50); print(ll, "| tail =", ll.tail.data)
print("popped", ll.pop_front(), "->", ll)

## 🔄 Part F: Reversing a Linked List

Reversing is the most famous linked-list interview question. We walk through the list once and **flip each `next` pointer** to point backwards. We need three references: `prev`, `cur` and `nxt`.

**Dry run**

```text
Start:   None   [10] -> [20] -> [30] -> None
         prev   cur
Step 1:  None <- [10]   [20] -> [30] -> None
                 prev   cur
Step 2:  None <- [10] <- [20]   [30] -> None
                         prev   cur
Step 3:  None <- [10] <- [20] <- [30]      head = prev
```

In [ ]:
def reverse(self):
    prev, cur = None, self.head
    self.tail = self.head
    while cur:
        nxt = cur.next             # 1. save the rest of the list
        cur.next = prev            # 2. flip the pointer
        prev = cur                 # 3. move prev forward
        cur = nxt                  # 4. move cur forward
    self.head = prev

LinkedList.reverse = reverse
demo = LinkedList()
for x in [10, 20, 30, 40]:
    demo.push_back(x)
demo.reverse()
print(demo)

Time O(n), extra space O(1). Notice how step 1 is essential: if you flip the pointer first, you lose the rest of the list forever.

## 🐢🐇 Part G: Slow and Fast Pointers

### Find the Middle in One Pass

Move `slow` one step and `fast` two steps at a time. When `fast` reaches the end, `slow` is at the middle.

In [ ]:
def middle(self):
    slow = fast = self.head
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
    return slow.data if slow else None

LinkedList.middle = middle
m = LinkedList()
for x in [1, 2, 3, 4, 5]:
    m.push_back(x)
print("middle of", m, "is", m.middle())

### Detect a Cycle (Floyd’s Tortoise and Hare)

If some node’s `next` points back to an earlier node, traversal would loop forever. With slow and fast pointers, if there is a cycle the fast pointer will eventually **meet** the slow one, like a faster runner lapping a slower runner on a circular track.

In [ ]:
def has_cycle(head):
    slow = fast = head
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
        if slow is fast:
            return True
    return False

nodes = [Node(i) for i in range(5)]
for i in range(4):
    nodes[i].next = nodes[i + 1]
print("cycle?", has_cycle(nodes[0]))
nodes[4].next = nodes[2]           # create a cycle 4 -> 2
print("cycle?", has_cycle(nodes[0]))

## ⚙️ C++ Corner

In C++, nodes are created with `new` and must be freed with `delete`. Forgetting to free them causes **memory leaks**.

**C++**

```cpp
#include <iostream>
using namespace std;

struct Node {
    int data;
    Node* next;
    Node(int d) : data(d), next(nullptr) {}
};

class LinkedList {
    Node* head = nullptr;
public:
    void pushFront(int d) { Node* n = new Node(d); n->next = head; head = n; }
    bool remove(int target) {
        Node** cur = &head;                 // pointer to the link we may change
        while (*cur && (*cur)->data != target) cur = &((*cur)->next);
        if (!*cur) return false;
        Node* dead = *cur;
        *cur = dead->next;
        delete dead;                        // free memory
        return true;
    }
    void reverse() {
        Node *prev = nullptr, *cur = head;
        while (cur) { Node* nxt = cur->next; cur->next = prev; prev = cur; cur = nxt; }
        head = prev;
    }
    void print() {
        for (Node* c = head; c; c = c->next) cout << c->data << " -> ";
        cout << "NULL" << endl;
    }
    ~LinkedList() { while (head) { Node* t = head; head = head->next; delete t; } }
};

int main() {
    LinkedList ll;
    for (int x : {40, 30, 20, 10}) ll.pushFront(x);
    ll.print(); ll.remove(30); ll.reverse(); ll.print();
}
```

## 🧪 Lab Tasks
Difficulty: 🟢 Easy · 🟡 Medium · 🔴 Hard

### Task 1 🟢 Build and Print
The notebook gives helper functions `from_list(values)` (builds a list and returns its head) and `to_list(head)`.

Write `length(head)` and `sum_values(head)` iteratively, and `length_rec(head)` **recursively**.

```text
head = from_list([4, 8, 15]) → length 3, sum 27
```

In [ ]:
class Node:
    def __init__(self, data, next=None):
        self.data = data
        self.next = next

def from_list(values):
    head = None
    for v in reversed(values):
        head = Node(v, head)
    return head

def to_list(head):
    out = []
    while head:
        out.append(head.data)
        head = head.next
    return out

def length(head):
    # TODO
    pass

def sum_values(head):
    # TODO
    pass

def length_rec(head):
    # TODO: base case None -> 0
    pass

In [ ]:
# Test cell: run after your solution
h = from_list([4, 8, 15])
assert length(h) == 3 and sum_values(h) == 27 and length_rec(h) == 3
assert length(None) == 0 and sum_values(None) == 0 and length_rec(None) == 0
print("✅ Task 1 passed")

### Task 2 🟢 Remove All Occurrences
Write `remove_all(head, value)` that deletes **every** node containing `value` and returns the (possibly new) head. Hint: a **dummy node** placed before the head removes the special case of deleting the head.

```text
[7, 1, 7, 2, 7] remove 7 → [1, 2]
```

In [ ]:
def remove_all(head, value):
    dummy = Node(0, head)
    prev = dummy
    # TODO: skip nodes equal to value
    return dummy.next

In [ ]:
# Test cell: run after your solution
assert to_list(remove_all(from_list([7, 1, 7, 2, 7]), 7)) == [1, 2]
assert to_list(remove_all(from_list([7, 7]), 7)) == []
assert to_list(remove_all(from_list([1, 2]), 9)) == [1, 2]
assert remove_all(None, 1) is None
print("✅ Task 2 passed")

### Task 3 🟡 Reverse (Iterative and Recursive)
Write `reverse_iter(head)` and `reverse_rec(head)`, each returning the new head. For the recursive version: reverse the rest, then make the second node point back to the first.

```text
[1, 2, 3, 4] → [4, 3, 2, 1]
```

In [ ]:
def reverse_iter(head):
    prev = None
    # TODO: prev / cur / nxt
    return prev

def reverse_rec(head):
    # TODO: base case; reverse the rest; fix two pointers
    pass

In [ ]:
# Test cell: run after your solution
for f in (reverse_iter, reverse_rec):
    assert to_list(f(from_list([1, 2, 3, 4]))) == [4, 3, 2, 1]
    assert to_list(f(from_list([9]))) == [9]
    assert f(None) is None
print("✅ Task 3 passed")

### Task 4 🟡 Merge Two Sorted Lists
Two sorted linked lists of roll numbers must be merged into one sorted list by **re-linking the existing nodes** (do not create new nodes except one dummy). Return the head.

```text
[1, 4, 9] + [2, 3, 10] → [1, 2, 3, 4, 9, 10]
```

In [ ]:
def merge_lists(a, b):
    dummy = tail = Node(0)
    # TODO: attach the smaller front node to tail each time
    return dummy.next

In [ ]:
# Test cell: run after your solution
assert to_list(merge_lists(from_list([1, 4, 9]), from_list([2, 3, 10]))) == [1, 2, 3, 4, 9, 10]
assert to_list(merge_lists(None, from_list([1]))) == [1]
assert merge_lists(None, None) is None
n1 = from_list([5]); n2 = from_list([1])
out = merge_lists(n1, n2); assert out is n2 and out.next is n1
print("✅ Task 4 passed")

### Task 5 🟡 Remove the N-th Node from the End
In **one pass**, remove the n-th node from the end and return the head. Hint: move a `fast` pointer n steps ahead, then move `fast` and `slow` together; use a dummy node.

```text
[1, 2, 3, 4, 5], n = 2 → [1, 2, 3, 5]
```

In [ ]:
def remove_nth_from_end(head, n):
    dummy = Node(0, head)
    fast = slow = dummy
    # TODO: gap of n between fast and slow, then move together
    return dummy.next

In [ ]:
# Test cell: run after your solution
assert to_list(remove_nth_from_end(from_list([1, 2, 3, 4, 5]), 2)) == [1, 2, 3, 5]
assert to_list(remove_nth_from_end(from_list([1]), 1)) == []
assert to_list(remove_nth_from_end(from_list([1, 2]), 2)) == [2]
print("✅ Task 5 passed")

### Task 6 🔴 Palindrome Linked List
Check whether a linked list reads the same forwards and backwards in **O(n) time and O(1) extra space**: find the middle (slow/fast), reverse the second half, compare the two halves, and (good practice) restore the list.

```text
[1, 2, 2, 1] → True,  [1, 2, 3] → False
```

In [ ]:
def is_palindrome_list(head):
    # TODO 1: find the middle with slow/fast
    # TODO 2: reverse the second half (you may use reverse_iter)
    # TODO 3: compare both halves, then restore
    pass

In [ ]:
# Test cell: run after your solution
h = from_list([1, 2, 2, 1]); assert is_palindrome_list(h) is True and to_list(h) == [1, 2, 2, 1]
assert is_palindrome_list(from_list([1, 2, 3])) is False
assert is_palindrome_list(from_list([1, 2, 3, 2, 1])) is True
assert is_palindrome_list(None) is True
print("✅ Task 6 passed")

### Task 7 🔴 Add Two Big Numbers
Very large numbers (bigger than any built-in integer type in C++) can be stored as linked lists of digits in **reverse order**. Add two such numbers and return the result as a linked list.

This is how arbitrary-precision arithmetic (used in cryptography and Python’s own big integers) works.

```text
342 + 465: [2, 4, 3] + [5, 6, 4] → [7, 0, 8]  (= 807)
```

In [ ]:
def add_numbers(a, b):
    dummy = tail = Node(0)
    carry = 0
    # TODO: add digit by digit with carry, like you did in school
    return dummy.next

In [ ]:
# Test cell: run after your solution
assert to_list(add_numbers(from_list([2, 4, 3]), from_list([5, 6, 4]))) == [7, 0, 8]
assert to_list(add_numbers(from_list([9, 9, 9]), from_list([1]))) == [0, 0, 0, 1]
assert to_list(add_numbers(from_list([0]), from_list([0]))) == [0]
x, y = 98765432109876543210, 12345678901234567890
dx = from_list([int(c) for c in str(x)[::-1]]); dy = from_list([int(c) for c in str(y)[::-1]])
assert int("".join(map(str, to_list(add_numbers(dx, dy))[::-1]))) == x + y
print("✅ Task 7 passed")

## 🔬 Bonus: Array vs Linked List, Insert at the Front

In [ ]:
# Bonus: run after completing the tasks
import time

class N:
    __slots__ = ("data", "next")
    def __init__(self, d, n=None):
        self.data, self.next = d, n

for n in [10_000, 50_000, 100_000]:
    t = time.perf_counter(); arr = []
    for i in range(n):
        arr.insert(0, i)                  # O(n) each -> O(n^2) total
    t_arr = time.perf_counter() - t

    t = time.perf_counter(); head = None
    for i in range(n):
        head = N(i, head)                 # O(1) each -> O(n) total
    t_ll = time.perf_counter() - t
    print(f"n={n:>7,}: list.insert(0) {t_arr:.3f}s | linked list push_front {t_ll:.3f}s")

---
## 🎉 Lab 08 Complete!

Next: **Lab 09: Doubly and Circular Linked Lists**

✅ Upload this notebook to your GitHub repository `DSA-Labs/Lab08/`.

---
*Prepared by **Muhammad Ramzan**, MPhil Scholar (AI), PUCIT, University of the Punjab*<br>
🔗 [GitHub](https://github.com/RamzanPUCIT) · [LinkedIn](https://www.linkedin.com/in/ramzan-ai) · [ResearchGate](https://www.researchgate.net/profile/Muhammad-Ramzan-105)